# 06 — Feature selection (fold-safe)

**Purpose.** Decide *which features the final model uses*, with evidence. Three questions, each answered by a rule written down **before** the results exist:

1. **Do the three literature-inspired features help?** (`Seronegative`, `ESR×CRP`, `RF×Anti-CCP`)
2. **Do the three engineered features help?** (ESR+CRP composite, C3/C4 ratio, autoantibody sum)
3. **Can a smaller feature set do just as well?** (top-k features by permutation importance, selected *inside* each fold)

**Reads:** `../data/raw/dataset.xlsx`, `../results/imb_summary.csv` · **Writes:** `../results/fs_*.csv`, `fs_oof.npz`

**Setting inherited from notebook 05:** Random Forest and XGBoost, **no class weighting** (the pre-specified winner), the same 5 folds as notebooks 04 and 05.

**Design**
* Selection is done *inside* every fold, never on the whole training set.
* Permutation importance is computed on **inner validation data**: each fold's training part is split 3 ways, the model is trained on 2/3 and importance is measured on the unseen 1/3 (importance measured on training rows is biased toward features the model memorised). The outer validation patients are never used for ranking.
* Different decision rules for *adding* and *dropping* features: adding a feature must be shown to help (superiority); dropping features must only be shown to **not hurt** (non-inferiority, margin 0.003 macro-F1), because the benefit of a smaller set is simplicity, not accuracy.
* `Seronegative` uses conventional cut-offs because the dataset's own RF / anti-CCP positivity cut-offs are undocumented (see caveats below).

**Candidate features (20 in total)**

| Group | Features | Count |
|---|---|---|
| Raw measurements | Age, Gender, ESR, CRP, RF, Anti-CCP, HLA-B27, ANA, Anti-Ro, Anti-La, Anti-dsDNA, Anti-Sm, C3, C4 | 14 |
| Engineered (notebook 02) | Inflammation_Score, C3_C4_Ratio, Autoantibody_Count | 3 |
| Literature-inspired (new) | Seronegative, ESR_CRP_Product, RF_AntiCCP_Product | 3 |

The two missingness flags are excluded for good (notebooks 01b and 04).

**Honest caveats**
* `Seronegative` = RF < 14 **and** anti-CCP < 20 (common lab positivity thresholds). It will *not* reproduce the 3.2% subgroup of Elmesiry et al. because their cut-offs are unknown to us; the prevalence is printed in section 3.
* Tree models can already learn thresholds and interactions, so these three features are *a priori* unlikely to add much. That is exactly why they are tested instead of assumed.
* Permutation importance is diluted among correlated features (ESR, CRP, their sum and their product; the six antibody markers and their sum). A low importance means "redundant given the others", not "useless on its own".

**Decision rules (pre-specified).** Applied to *both* models; a feature group or subset is accepted only if **both** models agree.
* **Rule 1 — literature features:** accept `all20` over `ext_noflags` only if, for both models, the 95% paired-bootstrap interval of the macro-F1 gain excludes 0 on the positive side **and** it wins in ≥ 4 of 5 folds.
* **Rule 2 — engineered features:** accept `ext_noflags` over `raw` under the same superiority criterion (otherwise the simpler `raw` set is preferred).
* **Reference set** = `all20` if Rule 1 passes, else `ext_noflags` if Rule 2 passes, else `raw`.
* **Rule 3 — smaller set:** a top-k set (k = 8, 10, 12, 14; fewer features than the reference) is *non-inferior* if, for both models, the lower end of the interval of (top-k − reference) is above **−0.003**. The smallest non-inferior k is adopted; if none qualifies, the reference stays. (If a top-k set turns out significantly *better* than the reference, that is reported as a finding but not auto-adopted: top-k lists may include features the reference excludes, so it would contradict Rules 1–2 and needs a closer look.)

## 0. Setup

**What this does:** imports libraries, loads the shared helpers (preprocessing, metrics), prints versions.

In [1]:
import sys, os, time, warnings
sys.path.append('..')

import numpy as np
import pandas as pd
import sklearn, xgboost
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

from src.preprocessing.cv_pipeline import (preprocess_fold, load_cv_pool, RAW_FEATURES, EXT_NO_FLAGS,
                                           LIT_FEATURES, SERO_RF_CUTOFF, SERO_CCP_CUTOFF)
from src.evaluation.metrics import (macro_f1_fast, per_class_prf, brier_multiclass, ece_top_label,
                                    holm, bootstrap_f1_matrix)

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| xgboost", xgboost.__version__)
os.makedirs('../results', exist_ok=True)

pandas 3.0.5 | numpy 2.5.2 | scikit-learn 1.9.0 | xgboost 3.4.1


## 1. Load the CV pool and the 5 folds

**What this does:** loads the 9,668 training patients (never the test set), verifies they are identical to notebook 02's training set, and recreates the same 5 stratified folds as notebooks 04 and 05 (`random_state=42`).

In [2]:
pool = load_cv_pool("../data/raw/dataset.xlsx")
ref_idx = pd.read_csv('../data/processed/y_train.csv', index_col=0).index
assert pool.shape == (9668, 17) and pool.index.equals(ref_idx)
y_global = LabelEncoder().fit_transform(pool['Disease'])
classes = sorted(pool['Disease'].unique()); K = len(classes)
AS, REA = classes.index('Ankylosing Spondylitis'), classes.index('Reactive Arthritis')

N_SPLITS = 5
folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42).split(pool, pool['Disease']))
cover = np.zeros(len(pool), int)
for tr, va in folds:
    cover[va] += 1
assert (cover == 1).all()
print("pool OK:", pool.shape, "| fold sizes:", [(len(tr), len(va)) for tr, va in folds])

pool OK: (9668, 17) | fold sizes: [(7734, 1934), (7734, 1934), (7734, 1934), (7735, 1933), (7735, 1933)]


## 2. Define the feature sets and check the new preprocessing option

**What this does:** names the three fixed feature sets and verifies that switching on the literature features did **not** change anything else.

| Fixed set | Columns | Meaning |
|---|---|---|
| `raw` | 14 | raw measurements |
| `ext_noflags` | 17 | raw + 3 engineered |
| `all20` | 20 | `ext_noflags` + 3 literature-inspired |

**Check (important):** `preprocess_fold(..., literature_features=True)` must return exactly the same values for the first 19 columns as the default call used in notebooks 02–05. If this fails, earlier results are not comparable.

**Seronegative prevalence:** printed for the first fold's training part. If it is far from a few per cent, the conventional cut-offs differ from the original paper's (expected; see caveat above).

In [3]:
ALL20 = EXT_NO_FLAGS + LIT_FEATURES
FIXED = {'raw': RAW_FEATURES, 'ext_noflags': EXT_NO_FLAGS, 'all20': ALL20}
assert [len(v) for v in FIXED.values()] == [14, 17, 20]

tr0, va0 = folds[0]
a = preprocess_fold(pool.iloc[tr0], pool.iloc[va0])
b = preprocess_fold(pool.iloc[tr0], pool.iloc[va0], literature_features=True)
assert list(b[2].columns[:19]) == list(a[2].columns)
assert np.allclose(a[2].to_numpy(float), b[2].iloc[:, :19].to_numpy(float))
assert np.allclose(a[0].to_numpy(float), b[0].to_numpy(float))
print("literature_features=True leaves the first 19 columns and the raw set unchanged: OK")
print("extra columns:", list(b[2].columns[19:]))
print(f"Seronegative prevalence in fold-1 training part (RF<{SERO_RF_CUTOFF}, anti-CCP<{SERO_CCP_CUTOFF}): {b[2]['Seronegative'].mean():.1%}")

literature_features=True leaves the first 19 columns and the raw set unchanged: OK
extra columns: ['Seronegative', 'ESR_CRP_Product', 'RF_AntiCCP_Product']
Seronegative prevalence in fold-1 training part (RF<14.0, anti-CCP<20.0): 28.0%


## 3. Models, and the permutation-importance ranking

**Models:** exactly as in notebook 05 with no weighting (Random Forest 300 trees; XGBoost 300 rounds, depth 5, lr 0.1).

**Permutation importance (how the ranking is made).** Shuffle one feature's values among patients and see how much macro-F1 drops: a big drop means the model relied on that feature. To keep the ranking honest, inside each fold's *training part*:
1. split into 3 inner folds;
2. train on 2/3, measure the macro-F1 drop on the unseen 1/3 (3 shuffles per feature);
3. average over the 3 inner folds → one importance per feature, ranked.

The top-k lists (k = 8, 10, 12, 14) are the k best-ranked of the 20 candidates. Because the ranking is recomputed inside each fold, **different folds may select different features** — that variation is itself reported (stability, section 8).

In [4]:
models = {
    'Random Forest': lambda: RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'XGBoost':       lambda: XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                           objective='multi:softprob', eval_metric='mlogloss',
                                           random_state=42, n_jobs=-1),
}
K_LIST = [8, 10, 12, 14]
set_names = ['raw', 'ext_noflags', 'all20'] + [f'top{k}' for k in K_LIST]
keys = [f"{m}|{s}" for m in models for s in set_names]

def importance_ranking(make, X, y, n_repeats=3):
    imps = []
    for itr, iva in StratifiedKFold(3, shuffle=True, random_state=1).split(X, y):
        m = make().fit(X.iloc[itr], y[itr])
        r = permutation_importance(m, X.iloc[iva], y[iva], scoring='f1_macro',
                                   n_repeats=n_repeats, random_state=0, n_jobs=1)
        imps.append(r.importances_mean)
    return pd.Series(np.mean(imps, axis=0), index=X.columns).sort_values(ascending=False)

print(len(keys), "configurations x", N_SPLITS, "folds")

14 configurations x 5 folds


## 4. The cross-validation loop

**What this does, for each fold:**
1. Fit imputer/scalers on the training part only; build all 20 candidate columns (`literature_features=True`).
2. For each model: compute the importance ranking from the training part only (inner 3-fold), derive the four top-k lists, then train and score the **7 feature sets** (`raw`, `ext_noflags`, `all20`, `top8`, `top10`, `top12`, `top14`) on the same validation patients.
3. Store out-of-fold probabilities and the importance values.

**Runtime:** roughly 10–20 minutes. One progress line per fold.

**Checks per fold:** no missing values; model classes 0–6; encoded labels equal the global encoding; every top-k list has exactly k distinct features.

In [5]:
oof_proba = {k: np.zeros((len(pool), K)) for k in keys}
y_all = np.full(len(pool), -1)
rows, imp_rows, sel_rows = [], [], []

for fold, (tr, va) in enumerate(folds, 1):
    t_fold = time.time()
    ft, fv = pool.iloc[tr], pool.iloc[va]
    Xr_tr, Xr_va, Xe_tr, Xe_va, y_tr, y_va = preprocess_fold(ft, fv, literature_features=True)
    for Xd in (Xr_tr, Xr_va, Xe_tr, Xe_va):
        assert Xd.isna().sum().sum() == 0
    assert (y_va.to_numpy() == y_global[va]).all() and (y_tr.to_numpy() == y_global[tr]).all()
    ytr, yva = y_tr.to_numpy(), y_va.to_numpy()
    y_all[va] = yva
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        for mname, make in models.items():
            rank = importance_ranking(make, Xe_tr[ALL20], ytr)
            for r_i, (feat, val) in enumerate(rank.items(), 1):
                imp_rows.append({'fold': fold, 'model': mname, 'feature': feat, 'importance': val, 'rank': r_i})
            sets = {'raw': (Xr_tr, Xr_va, RAW_FEATURES), 'ext_noflags': (Xe_tr, Xe_va, EXT_NO_FLAGS), 'all20': (Xe_tr, Xe_va, ALL20)}
            for k in K_LIST:
                cols = list(rank.index[:k]); assert len(set(cols)) == k
                sets[f'top{k}'] = (Xe_tr, Xe_va, cols)
                sel_rows.append({'fold': fold, 'model': mname, 'k': k, 'features': ', '.join(cols)})
            for sname in set_names:
                Xt_all, Xv_all, cols = sets[sname]
                m = make().fit(Xt_all[cols], ytr)
                assert list(m.classes_) == list(range(K))
                P = m.predict_proba(Xv_all[cols]); pred = P.argmax(1)
                oof_proba[f"{mname}|{sname}"][va] = P
                pr, rc, f1 = per_class_prf(yva, pred, K)
                rows.append({'fold': fold, 'model': mname, 'set': sname, 'key': f"{mname}|{sname}", 'n_features': len(cols),
                             'macro_f1': f1_score(yva, pred, average='macro'), 'balanced_accuracy': rc.mean(),
                             'AS_f1': f1[AS], 'ReA_f1': f1[REA]})
    print(f"fold {fold}/{N_SPLITS} done in {time.time() - t_fold:.0f}s")

fold_df = pd.DataFrame(rows); imp_df = pd.DataFrame(imp_rows); sel_df = pd.DataFrame(sel_rows)
oof_pred = {k: v.argmax(1) for k, v in oof_proba.items()}
print("rows in fold table:", len(fold_df))

fold 1/5 done in 53s
fold 2/5 done in 48s
fold 3/5 done in 49s
fold 4/5 done in 50s
fold 5/5 done in 52s
rows in fold table: 70


## 5. Integrity checks and reproduction of notebook 05

**What this does:**
* every patient has a label and every probability row sums to 1;
* the `raw` and `ext_noflags` unweighted results must **reproduce notebook 05's `none` strategy exactly** (same folds, same preprocessing, same models, same columns). This proves the literature-feature option and the new code path changed nothing for the `raw` and `ext_noflags` sets.

**If any difference exceeds 0.0005, stop: the new code path would have changed the `raw` / `ext_noflags` results and nothing below could be trusted.**

In [6]:
assert (y_all == y_global).all()
for k, P in oof_proba.items():
    assert np.allclose(P.sum(1), 1.0, atol=1e-6), k

ref = pd.read_csv('../results/imb_summary.csv', index_col=0)['macro_f1_mean']
gmean = fold_df.groupby('key').macro_f1.mean()
rep = pd.DataFrame([{'config': f"{m}|{s}", 'notebook_05_none': ref[f"{m}|{s}|none"], 'this_notebook': gmean[f"{m}|{s}"]}
                    for m in models for s in ('raw', 'ext_noflags')])
rep['diff'] = rep['this_notebook'] - rep['notebook_05_none']
print(rep.round(5).to_string(index=False))
print("\nmax |diff| =", rep['diff'].abs().max().round(6))
if rep['diff'].abs().max() > 0.0005:
    print("WARNING: does not reproduce notebook 05 - investigate before trusting anything below.")

                   config  notebook_05_none  this_notebook  diff
        Random Forest|raw           0.82303        0.82303   0.0
Random Forest|ext_noflags           0.82452        0.82452   0.0
              XGBoost|raw           0.81741        0.81741   0.0
      XGBoost|ext_noflags           0.81746        0.81746   0.0

max |diff| = 0.0


## 6. Results table

**What this does:** one row per (model, feature set), sorted by mean fold macro-F1.

* `n_features` — size of the set (for top-k: k).
* `macro_f1_mean ± std` — average and spread over the 5 folds (main number).
* `oof_macro_f1`, `ci_low`, `ci_high` — pooled out-of-fold score with a 95% bootstrap interval.
* `AS_f1`, `ReA_f1` — the two problem classes; `balanced_acc`, `brier`, `ece` as before.

**How to read it:** differences between neighbouring rows are usually smaller than the interval; the rules in section 9 decide which differences count.

In [7]:
keys_b, F = bootstrap_f1_matrix(y_all, oof_pred, B=2000, seed=42)
kpos = {k: j for j, k in enumerate(keys_b)}
gk = fold_df.groupby('key')
rows_s = []
for k in keys:
    m, s = k.split('|')
    pr, rc, f1 = per_class_prf(y_all, oof_pred[k], K)
    rows_s.append({'model': m, 'set': s, 'n_features': int(gk.n_features.first()[k]),
                   'macro_f1_mean': gk.macro_f1.mean()[k], 'macro_f1_std': gk.macro_f1.std()[k],
                   'oof_macro_f1': macro_f1_fast(y_all, oof_pred[k]),
                   'ci_low': np.percentile(F[:, kpos[k]], 2.5), 'ci_high': np.percentile(F[:, kpos[k]], 97.5),
                   'AS_f1': f1[AS], 'ReA_f1': f1[REA], 'balanced_acc': rc.mean(),
                   'brier': brier_multiclass(y_all, oof_proba[k]), 'ece': ece_top_label(y_all, oof_proba[k])})
summary = pd.DataFrame(rows_s, index=keys).sort_values('macro_f1_mean', ascending=False)
summary.round(4)

,model,set,n_features,macro_f1_mean,macro_f1_std,oof_macro_f1,ci_low,ci_high,AS_f1,ReA_f1,balanced_acc,brier,ece
Random Forest|all20,Random Forest,all20,20,0.8249,0.0132,0.8249,0.8162,0.8332,0.6647,0.6631,0.8166,0.2167,0.0263
Random Forest|ext_noflags,Random Forest,ext_noflags,17,0.8245,0.0098,0.8245,0.8154,0.8329,0.6667,0.6622,0.8156,0.2150,0.0246
Random Forest|raw,Random Forest,raw,14,0.8230,0.0070,0.8230,0.8139,0.8316,0.6623,0.6563,0.8114,0.2196,0.0370
Random Forest|top14,Random Forest,top14,14,0.8226,0.0101,0.8226,0.8139,0.8313,0.6656,0.6596,0.8147,0.2158,0.0210
XGBoost|all20,XGBoost,all20,20,0.8177,0.0055,0.8178,0.8092,0.8263,0.6679,0.6459,0.8123,0.2304,0.0564
XGBoost|ext_noflags,XGBoost,ext_noflags,17,0.8175,0.0083,0.8175,0.8091,0.8257,0.6630,0.6503,0.8119,0.2307,0.0583
XGBoost|raw,XGBoost,raw,14,0.8174,0.0057,0.8175,0.8090,0.8258,0.6673,0.6439,0.8108,0.2308,0.0575
Random Forest|top12,Random Forest,top12,12,0.8171,0.0088,0.8171,0.8083,0.8259,0.6568,0.6456,0.8101,0.2179,0.0185
Random Forest|top10,Random Forest,top10,10,0.8124,0.0072,0.8124,0.8035,0.8215,0.6525,0.6357,0.8033,0.2267,0.0190
XGBoost|top12,XGBoost,top12,12,0.8110,0.0069,0.8112,0.8026,0.8198,0.6500,0.6376,0.8065,0.2321,0.0620


## 7. Permutation importance of all 20 candidates

**What this does:** for each model, the mean importance (macro-F1 drop when the feature is shuffled) over the 5 folds, its spread across folds, and its average rank.

**How to read it:**
* A large `mean_importance` with a small `std` = a feature the model consistently relies on.
* `top10_in_folds` (0–5) = in how many folds the feature was in the top 10. **A feature that is in the top 10 in 5/5 folds is stable; 2/5 means the ranking is unreliable.**
* Importances near 0 (or slightly negative) mean the feature adds nothing *beyond the others* — typical for products and sums of features that are already present.

In [8]:
stab = imp_df.groupby(['model', 'feature']).agg(mean_importance=('importance', 'mean'), std=('importance', 'std'),
                                                mean_rank=('rank', 'mean'),
                                                top10_in_folds=('rank', lambda r: int((r <= 10).sum()))).reset_index()
group = {f: 'raw' for f in RAW_FEATURES}
group.update({f: 'engineered' for f in ['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']})
group.update({f: 'literature' for f in LIT_FEATURES})
stab['group'] = stab['feature'].map(group)
for m in models:
    print(f"\n=== {m} ===")
    print(stab[stab.model == m].sort_values('mean_importance', ascending=False)
          .drop(columns='model').round(4).to_string(index=False))


=== Random Forest ===
           feature  mean_importance    std  mean_rank  top10_in_folds      group
               ESR           0.0503 0.0052        1.0               5        raw
           HLA-B27           0.0402 0.0048        2.2               5        raw
          Anti-CCP           0.0350 0.0023        3.4               5        raw
                RF           0.0348 0.0032        3.4               5        raw
                C3           0.0247 0.0017        5.4               5        raw
RF_AntiCCP_Product           0.0227 0.0008        6.2               5 literature
           Anti-Ro           0.0208 0.0029        7.4               5        raw
           Anti-La           0.0199 0.0018        8.0               5        raw
                C4           0.0185 0.0009        9.4               5        raw
               ANA           0.0182 0.0031        9.4               3        raw
Inflammation_Score           0.0176 0.0014       10.2               2 engineered
   ES

## 8. Which features did each fold select?

**What this does:** shows, for the top-10 lists, which features were picked in each fold, so you can see how stable the selection is. (Lists for k = 8, 12, 14 are saved to `fs_selected_features.csv`.)

In [9]:
for m in models:
    print(f"\n=== {m}: top-10 selected in each fold ===")
    for _, r in sel_df[(sel_df.model == m) & (sel_df.k == 10)].iterrows():
        print(f"fold {r['fold']}: {r['features']}")


=== Random Forest: top-10 selected in each fold ===
fold 1: ESR, HLA-B27, Anti-CCP, RF, Anti-Ro, RF_AntiCCP_Product, C3, Anti-La, ANA, C4
fold 2: ESR, RF, HLA-B27, Anti-CCP, C3, RF_AntiCCP_Product, Anti-La, C4, Anti-Ro, Inflammation_Score
fold 3: ESR, HLA-B27, Anti-CCP, RF, C3, RF_AntiCCP_Product, ANA, Anti-La, C4, Anti-Ro
fold 4: ESR, HLA-B27, Anti-CCP, RF, C3, Anti-Ro, RF_AntiCCP_Product, Anti-La, ANA, C4
fold 5: ESR, HLA-B27, RF, Anti-CCP, C3, RF_AntiCCP_Product, Anti-Ro, Inflammation_Score, Anti-La, C4

=== XGBoost: top-10 selected in each fold ===
fold 1: ESR, Anti-CCP, HLA-B27, RF, Inflammation_Score, CRP, Anti-Ro, Anti-La, C4, C3
fold 2: ESR, RF, Anti-CCP, HLA-B27, Anti-La, Inflammation_Score, CRP, C3, C4, Anti-Ro
fold 3: ESR, RF, Anti-CCP, HLA-B27, Inflammation_Score, Anti-La, Anti-Ro, C3, C4, CRP
fold 4: ESR, RF, Anti-CCP, HLA-B27, CRP, Anti-Ro, Anti-La, C3, C4, Inflammation_Score
fold 5: ESR, Anti-CCP, HLA-B27, RF, CRP, Anti-La, C3, Anti-Ro, Inflammation_Score, C4


## 9. Applying the pre-specified rules

**What this does:** runs Rules 1–3 from the top of the notebook, for **both** models, and prints every comparison used.

Columns: `diff` = pooled out-of-fold macro-F1 of A minus B; `ci_low`/`ci_high` = 95% paired-bootstrap interval; `folds_A_better` = folds (of 5) where A beats B. For Rules 1–2 a comparison **passes** if `ci_low > 0` and `folds_A_better ≥ 4`; for Rule 3 it passes (non-inferior) if `ci_low > -0.003`.

In [10]:
piv = fold_df.pivot(index='fold', columns='key', values='macro_f1')
def compare(A, B):
    d = F[:, kpos[A]] - F[:, kpos[B]]
    fd = piv[A] - piv[B]
    return {'A': A, 'B': B, 'diff': macro_f1_fast(y_all, oof_pred[A]) - macro_f1_fast(y_all, oof_pred[B]),
            'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5), 'folds_A_better': int((fd > 0).sum())}

# Rule 1 and Rule 2 (superiority)
r12 = []
for m in models:
    r12.append({**compare(f"{m}|all20", f"{m}|ext_noflags"), 'rule': '1: literature features'})
    r12.append({**compare(f"{m}|ext_noflags", f"{m}|raw"),   'rule': '2: engineered features'})
r12 = pd.DataFrame(r12)
r12['passes'] = (r12['ci_low'] > 0) & (r12['folds_A_better'] >= 4)
print(r12.round(4).to_string(index=False))

lit_keep = bool(r12[r12.rule.str.startswith('1')]['passes'].all())
eng_keep = bool(r12[r12.rule.str.startswith('2')]['passes'].all())
reference = 'all20' if lit_keep else ('ext_noflags' if eng_keep else 'raw')
n_ref = {'raw': 14, 'ext_noflags': 17, 'all20': 20}[reference]
print(f"\nRule 1 (literature features help in BOTH models): {lit_keep}")
print(f"Rule 2 (engineered features help in BOTH models):  {eng_keep}")
print(f"REFERENCE SET: {reference} ({n_ref} features)")

# Rule 3 (non-inferiority of smaller top-k sets)
r3 = []
for k in K_LIST:
    for m in models:
        r3.append({**compare(f"{m}|top{k}", f"{m}|{reference}"), 'k': k, 'model': m})
r3 = pd.DataFrame(r3)
r3['non_inferior'] = r3['ci_low'] > -0.003
r3['superior'] = r3['ci_low'] > 0
print("\nRule 3: top-k vs reference (non-inferiority margin -0.003)")
print(r3.drop(columns=['A', 'B']).round(4).to_string(index=False))
ok_k = [k for k in K_LIST if k < n_ref and r3[r3.k == k]['non_inferior'].all()]
final_set = f"top{min(ok_k)}" if ok_k else reference
print(f"\nSmallest k non-inferior in BOTH models (k < {n_ref}): {min(ok_k) if ok_k else 'none'}")
print(f"FINAL FEATURE-SET DECISION: {final_set}")
sup_k = [k for k in K_LIST if r3[r3.k == k]['superior'].all()]
if sup_k:
    print(f"NOTE: top-k sets with k in {sup_k} are significantly BETTER than the reference in both models. "
          "Because top-k lists may include features outside the reference set, this means some excluded features "
          "do carry signal. The rules do not auto-adopt this; investigate before adopting.")

                        A                         B   diff  ci_low  ci_high  folds_A_better                   rule  passes
      Random Forest|all20 Random Forest|ext_noflags 0.0004 -0.0033   0.0041               4 1: literature features   False
Random Forest|ext_noflags         Random Forest|raw 0.0015 -0.0026   0.0055               2 2: engineered features   False
            XGBoost|all20       XGBoost|ext_noflags 0.0003 -0.0044   0.0048               2 1: literature features   False
      XGBoost|ext_noflags               XGBoost|raw 0.0000 -0.0050   0.0052               3 2: engineered features   False

Rule 1 (literature features help in BOTH models): False
Rule 2 (engineered features help in BOTH models):  False
REFERENCE SET: raw (14 features)

Rule 3: top-k vs reference (non-inferiority margin -0.003)
   diff  ci_low  ci_high  folds_A_better  k         model  non_inferior  superior
-0.0275 -0.0343  -0.0208               0  8 Random Forest         False     False
-0.0349 -0.042

## 10. Save the results

**What this does:** writes all tables and the out-of-fold probabilities to `../results/` (notebooks 04 and 05 outputs are untouched).

**What happens next.** If the decision is a fixed set (`raw`, `ext_noflags` or `all20`), notebook 07 (hyper-parameter tuning) uses exactly those columns. If it is a `top-k` set, the selection step must be repeated *inside* every tuning fold (the final feature list is then recomputed on the full training data); nothing in notebook 07 may reuse the per-fold lists from here.

In [11]:
fold_df.drop(columns=['key']).to_csv('../results/fs_fold_scores.csv', index=False)
summary.to_csv('../results/fs_summary.csv')
stab.to_csv('../results/fs_importance_stability.csv', index=False)
sel_df.to_csv('../results/fs_selected_features.csv', index=False)
pd.concat([r12.assign(kind='superiority'), r3.assign(kind='non_inferiority')], ignore_index=True).to_csv('../results/fs_rule_comparisons.csv', index=False)
np.savez_compressed('../results/fs_oof.npz', y=y_all, classes=np.array(classes),
                    **{k.replace('|', '__').replace(' ', '_'): v.astype(np.float32) for k, v in oof_proba.items()})
print(sorted(f for f in os.listdir('../results') if f.startswith('fs_')))

['fs_fold_scores.csv', 'fs_importance_stability.csv', 'fs_oof.npz', 'fs_rule_comparisons.csv', 'fs_selected_features.csv', 'fs_summary.csv']


## Summary — what this notebook establishes

* Whether the three literature-inspired features and the three engineered features earn a place (Rules 1–2), judged jointly on Random Forest and XGBoost.
* Which features are consistently important and which are redundant (section 7), and how stable fold-wise selection is (section 8).
* Whether a smaller set is statistically as good (Rule 3).
* The **feature-set decision for notebook 07** (printed at the end of section 9).